# Compréhension globale du projet
## 🎯 Objectif final
```
Créer un dataset d'images propre et homogène pour entraîner un modèle de reconnaissance de déchets (6 classes : cardboard, plastic, paper, glass, metal, trash).

📚 Les 14 parties de l'atelier
Partie	Sujet
0	Structure du projet
1	Exploration du dataset
2	Détection des images corrompues
3	Détection des images vides
4	Détection des différences de résolution
5	Détection des différents canaux
6	Détection des doublons
7	Détection des images mal classées
8	Analyse du déséquilibre des classes
9	Redimensionnement
10	Uniformisation des canaux
11	Mise à l'échelle des pixels
12	Découpage Train/Validation/Test
13	Data Augmentation
14	Bonus
```

# Partie 1 – Exploration du dataset
## 🎯 Objectif
```
Lire toutes les images de chaque dossier et récupérer leurs informations :

Nom, classe, format, mode

Largeur, hauteur

Écart-type des pixels (pour détecter les images vides)

Nombre de canaux

Taille du fichier
```

## 📝 Explication imagée
# C'est comme faire l'inventaire d'un entrepôt :
```
On ouvre chaque carton (dossier)

On note ce qu'il contient (nom, taille, dimensions)

On repère les cartons abîmés (images corrompues)
```

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageStat
import warnings
warnings.filterwarnings('ignore')

# Chemins
BASE_DIR = '../data/raw'
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

print("✅ Bibliothèques importées")
print(f"📁 Dossier de base : {BASE_DIR}")
print(f"📂 Classes : {CLASSES}")

✅ Bibliothèques importées
📁 Dossier de base : ../data/raw
📂 Classes : ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


##  Fonction de détection d'image corrompue

In [4]:
def explorer_dataset(base_dir, classes):
    """Parcourt toutes les images et collecte leurs informations."""
    donnees = []
    
    for classe in classes:
        dossier = os.path.join(base_dir, classe)
        if not os.path.exists(dossier):
            print(f"⚠️ Dossier manquant : {dossier}")
            continue
        
        fichiers = os.listdir(dossier)
        print(f"📂 {classe} : {len(fichiers)} fichiers")
        
        for fichier in fichiers:
            chemin = os.path.join(dossier, fichier)
            
            # Ignorer les sous-dossiers
            if not os.path.isfile(chemin):
                continue
            
            # Informations de base
            info = {
                'nom': fichier,
                'classe': classe,
                'chemin': chemin,
                'taille_ko': round(os.path.getsize(chemin) / 1024, 2),
                'corrompue': False,
                'format': None,
                'mode': None,
                'largeur': None,
                'hauteur': None,
                'canaux': None,
                'ecart_type': None
            }
            
            # Vérifier si corrompue
            if est_corrompue(chemin):
                info['corrompue'] = True
                donnees.append(info)
                continue
            
            # Extraire les métadonnées
            try:
                with Image.open(chemin) as img:
                    info['format'] = img.format
                    info['mode'] = img.mode
                    info['largeur'], info['hauteur'] = img.size
                    info['canaux'] = len(img.getbands())
                    
                    # Écart-type des pixels (pour détecter images vides)
                    stat = ImageStat.Stat(img.convert('L'))
                    info['ecart_type'] = round(stat.stddev[0], 2)
            except Exception as e:
                info['corrompue'] = True
            
            donnees.append(info)
    
    return pd.DataFrame(donnees)

# Exécution
df = explorer_dataset(BASE_DIR, CLASSES)

print(f"\n✅ Exploration terminée : {len(df)} images analysées")
print(f"📊 Aperçu :")
df.head()

📂 cardboard : 169 fichiers
📂 glass : 188 fichiers
📂 metal : 149 fichiers
📂 paper : 252 fichiers
📂 plastic : 224 fichiers
📂 trash : 50 fichiers

✅ Exploration terminée : 1032 images analysées
📊 Aperçu :


,nom,classe,chemin,taille_ko,corrompue,format,mode,largeur,hauteur,canaux,ecart_type
0,cardboard1.jpg,cardboard,../data/raw\cardboard\cardboard1.jpg,16.93,False,JPEG,RGB,512.0,384.0,3.0,31.88
1,cardboard10.jpg,cardboard,../data/raw\cardboard\cardboard10.jpg,21.17,False,JPEG,RGB,512.0,384.0,3.0,38.80
2,cardboard100.jpg,cardboard,../data/raw\cardboard\cardboard100.jpg,14.54,False,JPEG,RGB,512.0,384.0,3.0,44.50
3,cardboard101.jpg,cardboard,../data/raw\cardboard\cardboard101.jpg,13.95,False,JPEG,RGB,512.0,384.0,3.0,68.56
4,cardboard102.jpg,cardboard,../data/raw\cardboard\cardboard102.jpg,17.59,False,JPEG,RGB,512.0,384.0,3.0,45.32


## Sauvegarde de l'audit

In [5]:
# Créer le dossier reports s'il n'existe pas
os.makedirs('../reports', exist_ok=True)

# Sauvegarder l'audit
df.to_csv('../reports/audit_images.csv', index=False)

print("✅ Audit sauvegardé dans '../reports/audit_images.csv'")
print(f"📊 Nombre total d'images : {len(df)}")
print(f"❌ Images corrompues : {df['corrompue'].sum()}")

✅ Audit sauvegardé dans '../reports/audit_images.csv'
📊 Nombre total d'images : 1032
❌ Images corrompues : 6


## Statistiques générales

In [6]:
print("=== STATISTIQUES GÉNÉRALES ===\n")

print(f"📊 Total d'images : {len(df)}")
print(f"❌ Images corrompues : {df['corrompue'].sum()}")

print("\n📂 Répartition par classe :")
print(df['classe'].value_counts())

print("\n🎨 Répartition par mode :")
print(df['mode'].value_counts())

print("\n📐 Répartition par nombre de canaux :")
print(df['canaux'].value_counts())

=== STATISTIQUES GÉNÉRALES ===

📊 Total d'images : 1032
❌ Images corrompues : 6

📂 Répartition par classe :
classe
paper        252
plastic      224
glass        188
cardboard    169
metal        149
trash         50
Name: count, dtype: int64

🎨 Répartition par mode :
mode
RGB     1006
RGBA      18
P          2
Name: count, dtype: int64

📐 Répartition par nombre de canaux :
canaux
3.0    1006
4.0      18
1.0       2
Name: count, dtype: int64
